# Aprofundamento - 3 fundos priorizados
Kinea Alpes Prev, Kinea Infra FII (KDIF11), Kinea High Yield CRI FII (KNHY11)

## 1. Setup

In [1]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

df_completo = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "universo_kinea_completo.csv")
df_concorrentes = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "fundos_concorrentes.csv")

FUNDOS_PRIORIZADOS = [
    "Kinea Alpes Prev XP Seg RF CP FICFI",
    "Kinea Infra FII (KDIF11)",
    "Kinea High Yield CRI FII (KNHY11)",
]
print("Fundos priorizados:", FUNDOS_PRIORIZADOS)

Fundos priorizados: ['Kinea Alpes Prev XP Seg RF CP FICFI', 'Kinea Infra FII (KDIF11)', 'Kinea High Yield CRI FII (KNHY11)']


## 2. Liquidez e custos - taxa de administração CVM dos concorrentes diretos do KNHY11

Universo amplo de concorrentes do KNHY11 filtrado por segmento CRI/Recebíveis,
com taxa de administração mensal reportada à CVM (convertida depois para % a.a.
na consolidação final, seção 5).

In [2]:
concorrentes_knhy11 = df_concorrentes[
    (df_concorrentes["referencia_fundo_kinea"] == "Kinea High Yield CRI FII (KNHY11)")
    & (~df_concorrentes["eh_fundo_kinea"])
].sort_values("patrimonio_liquido", ascending=False)

concorrentes_cri = concorrentes_knhy11[
    concorrentes_knhy11["nome"].str.contains("CRI|RECEBÍVEIS|HIGH YIELD|CRÉDITO", case=False, na=False, regex=True)
]
concorrentes_cri_validos = concorrentes_cri[concorrentes_cri["taxa_administracao_cvm_mensal_pct"].notna()].copy()
concorrentes_cri_validos["taxa_administracao_anual_pct_est"] = (
    concorrentes_cri_validos["taxa_administracao_cvm_mensal_pct"] * 1200
).round(2)

print(f"{len(concorrentes_cri_validos)} concorrentes CRI válidos "
      f"(de {len(concorrentes_knhy11)} concorrentes totais do KNHY11)")
print(f"Mediana da taxa a.a. estimada: {concorrentes_cri_validos['taxa_administracao_anual_pct_est'].median()}%")
concorrentes_cri_validos[["nome", "patrimonio_liquido", "taxa_administracao_anual_pct_est"]].head(10)

38 concorrentes CRI válidos (de 647 concorrentes totais do KNHY11)
Mediana da taxa a.a. estimada: 0.99%


,nome,patrimonio_liquido,taxa_administracao_anual_pct_est
17670,IRIDIUM CRI FII RL,2.898858e+09,1.03
17567,VALORA CRI CDI FII,1.432803e+09,1.04
17642,DEVANT RECEBÍVEIS IMOBILIÁRIOS FII,1.354867e+09,0.08
18096,INSTR PART DE DELIB CONJUNTA DE ALTERAÇÃO DO X...,1.265091e+09,0.00
18088,XP CDI 94 CRI JUN/27 FUNDO DE INVESTIMENTO IMO...,1.138259e+09,0.00
17933,JS CRÉDITO ESTRUTURADO FI IMOBILIARIO RESP LIM...,1.118366e+09,1.10
17607,VALORA CRI FII RL,1.068558e+09,0.92
17578,HABITAT RECEBÍVEIS PULVERIZADOS FII RL,7.663912e+08,0.00
18037,JS CRÉDITO ESTRUTURADO II FII IMOBILIÁRIO RL,7.414414e+08,0.00
18008,VALORA CRI PRÉ I MASTER FUNDO DE INVESTIMENTO ...,6.780465e+08,0.00


## 3. Coleta dos concorrentes diretos (XP) - Alpes Prev, KDIF11, KNHY11

Config com os 8 concorrentes diretos identificados manualmente, seguido do
scraper (mesmo script do pipeline principal, `xp_fund_scraper.py`).

In [3]:
import csv

concorrentes = [
    # Alpes Prev (Previdência Multimercado Crédito Privado)
    {"nome_referencia": "SPX Seahawk Icatu Previdencia FICFIM Multimercado Credito Privado",
     "url": "https://conteudos.xpi.com.br/previdencia-privada/spx-seahawk-icatu-previdencia-ficfi-multimercado-credito-privado/",
     "tipo_pagina": "previdencia", "status_confirmacao": "identificado_pendente",
     "referencia_fundo_kinea": "Kinea Alpes Prev"},
    {"nome_referencia": "Ibiuna ST Prev Icatu FIM Multimercado Credito Privado",
     "url": "https://conteudos.xpi.com.br/previdencia-privada/ibiuna-st-prev-icatu-fundo-de-investimento-multimercado-credito-privado/",
     "tipo_pagina": "previdencia", "status_confirmacao": "identificado_pendente",
     "referencia_fundo_kinea": "Kinea Alpes Prev"},
    {"nome_referencia": "Itau Sinfonia XP Seg Prev FIC Multimercado Credito Privado",
     "url": "https://conteudos.xpi.com.br/previdencia-privada/itau-sinfonia-xp-seg-prev-fundo-de-investimento-em-cotas-multimercado-credito-privado-responsabilidade-limitada/",
     "tipo_pagina": "previdencia", "status_confirmacao": "identificado_pendente",
     "referencia_fundo_kinea": "Kinea Alpes Prev"},
    # KDIF11 (FI-Infra / Debêntures Incentivadas)
    {"nome_referencia": "Sparta Debentures Incentivadas FIC FI-Infra RF",
     "url": "https://conteudos.xpi.com.br/fundos-de-investimento/sparta-debentures-incentivadas-firf-cp/",
     "tipo_pagina": "fundo_aberto", "status_confirmacao": "identificado_pendente",
     "referencia_fundo_kinea": "Kinea Infra FII (KDIF11)"},
    {"nome_referencia": "Itau Debentures Incentivadas FIF CIC em Infra",
     "url": "https://conteudos.xpi.com.br/fundos-de-investimento/itau-debentures-incentivadas-fc-incentivado-investimento-inf/",
     "tipo_pagina": "fundo_aberto", "status_confirmacao": "identificado_pendente",
     "referencia_fundo_kinea": "Kinea Infra FII (KDIF11)"},
    {"nome_referencia": "SulAmerica Infra FI Infra RF",
     "url": "https://conteudos.xpi.com.br/fundos-de-investimento/sulamerica-debentures-incentivadas-fim-cp/",
     "tipo_pagina": "fundo_aberto", "status_confirmacao": "identificado_pendente",
     "referencia_fundo_kinea": "Kinea Infra FII (KDIF11)"},
    # KNHY11 (FII CRI High Yield)
    {"nome_referencia": "Devant Recebiveis Imobiliarios FII (DEVA11)",
     "url": "https://conteudos.xpi.com.br/fundos-imobiliarios/devant-recebiveis-imobiliarios-fii-deva11/",
     "tipo_pagina": "fii", "status_confirmacao": "identificado_pendente",
     "referencia_fundo_kinea": "Kinea High Yield CRI FII (KNHY11)"},
    {"nome_referencia": "Valora CRI Indice de Preco FII (VGIP11)",
     "url": "https://conteudos.xpi.com.br/fundos-imobiliarios/valora-cri-indice-de-preco-fii-vgip11/",
     "tipo_pagina": "fii", "status_confirmacao": "identificado_pendente",
     "referencia_fundo_kinea": "Kinea High Yield CRI FII (KNHY11)"},
]

config_path = PROJECT_ROOT / "config" / "concorrentes_aprofundamento.csv"
with open(config_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(concorrentes[0].keys()))
    writer.writeheader()
    writer.writerows(concorrentes)
print("Config salvo em:", config_path, f"({len(concorrentes)} concorrentes)")

Config salvo em: /Users/julianamurakami/Downloads/case-kinea-bi/config/concorrentes_aprofundamento.csv (8 concorrentes)


In [4]:
import subprocess

resultado = subprocess.run(
    [sys.executable, "src/ingestion/xp_fund_scraper.py",
     "--input", "config/concorrentes_aprofundamento.csv",
     "--raw-dir", "data/raw/html_concorrentes",
     "--output", "data/raw/concorrentes_aprofundamento_raw.csv"],
    cwd=PROJECT_ROOT, capture_output=True, text=True,
)
print(resultado.stdout)
print(resultado.stderr)

KeyboardInterrupt: 

In [ ]:
df_conc_raw = pd.read_csv(PROJECT_ROOT / "data" / "raw" / "concorrentes_aprofundamento_raw.csv")
pd.set_option("display.max_colwidth", None)
df_conc_raw[["nome_referencia", "taxa_administracao", "objetivo", "benchmark", "erro"]]

,nome_referencia,taxa_administracao,objetivo,benchmark,erro
0,SPX Seahawk Icatu Previdencia FICFIM Multimercado Credito Privado,"0,80 %",NaN,NaN,NaN
1,Ibiuna ST Prev Icatu FIM Multimercado Credito Privado,"2,00 %",NaN,NaN,NaN
2,Itau Sinfonia XP Seg Prev FIC Multimercado Credito Privado,"0,64 %",NaN,CDI,NaN
3,Sparta Debentures Incentivadas FIC FI-Infra RF,"0,80 %",100% do CDI isento,CDI,NaN
4,Itau Debentures Incentivadas FIF CIC em Infra,"0,85 %",Retorno acima das Bs de referência da duration proxima ao IMA-B com risco de crédito,IMA-B,NaN
5,SulAmerica Infra FI Infra RF,"0,80 %","O objetivo do Fundo é acompanhar, a médio/longo prazo, a variação do IMA-B5.",IMA-B 5,NaN
6,Devant Recebiveis Imobiliarios FII (DEVA11),"1,00 %",NaN,NaN,NaN
7,Valora CRI Indice de Preco FII (VGIP11),"1,00 %",NaN,NaN,NaN


## 4. Sanity check - campos de Retorno e risco (`rr_*` / `rent_*`)

Confirma que os 3 fundos Kinea priorizados vieram com os campos novos do
scraper preenchidos (tipo de página determina se é `rr_*` ou `rent_*`).

In [ ]:
df_kinea_raw = pd.read_csv(PROJECT_ROOT / "data" / "raw" / "universo_kinea_raw.csv")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)

alvos = ["Kinea Alpes Prev", "Kinea Infra", "Kinea High Yield"]
mask = df_kinea_raw["nome_referencia"].str.contains("|".join(alvos), case=False, na=False)
sub = df_kinea_raw[mask]

cols_rr = [c for c in df_kinea_raw.columns if c.startswith("rr_")]
cols_rent = [c for c in df_kinea_raw.columns if c.startswith("rent_")]

print("=== tipo_pagina ===")
print(sub[["nome_referencia", "tipo_pagina"]])
print("\n=== rr_* (Risco e Retorno: fundo_aberto/previdencia) ===")
print(sub[["nome_referencia"] + cols_rr].to_string())
print("\n=== rent_* (Rentabilidade: fii) ===")
print(sub[["nome_referencia"] + cols_rent].to_string())

=== tipo_pagina ===
                        nome_referencia  tipo_pagina
6   Kinea Alpes Prev XP Seg RF CP FICFI  previdencia
9              Kinea Infra FII (KDIF11)          fii
12    Kinea High Yield CRI FII (KNHY11)          fii

=== rr_* (Risco e Retorno: fundo_aberto/previdencia) ===
                        nome_referencia rr_rentabilidade_12m rr_rentabilidade_24m rr_rentabilidade_36m rr_rentabilidade_desde_inicio rr_rentabilidade_no_ano rr_sharpe_12m rr_sharpe_24m rr_sharpe_36m rr_sharpe_desde_inicio rr_sharpe_no_ano rr_volatilidade_12m rr_volatilidade_24m rr_volatilidade_36m rr_volatilidade_desde_inicio rr_volatilidade_no_ano
6   Kinea Alpes Prev XP Seg RF CP FICFI              13,83 %                  NaN                  NaN                        61,59%                  8,35 %          0,60          2,51          3,06                   2,31             5,54               0,39%               0,33%               0,33%                        0,38%                  0,23%
9       

## 5. Liquidez e custos + Conteúdo e diferenciação - consolidação

Números já validados em diagnóstico anterior (ver `docs/log_ia.md`):
- Liquidez e custos: taxa Kinea sempre acima da mediana dos concorrentes diretos.
- Conteúdo e diferenciação: só o KDIF11 tem achado real (ficha 50% incompleta);
  Alpes Prev sem achado (limitação do template de Previdência da XP, não é
  falha específica da Kinea); KNHY11 sem achado (ficha 100% completa).

In [ ]:
def _pct_to_float(serie: pd.Series) -> pd.Series:
    """Converte coluna de porcentagem em texto ('0,80 %') para float (0.80)."""
    return pd.to_numeric(
        serie.astype(str).str.replace("%", "").str.replace(",", ".").str.strip(),
        errors="coerce",
    )


def filtrar(df: pd.DataFrame, *termos: str) -> pd.DataFrame:
    """Filtra nome_referencia por qualquer um dos termos dados (case-insensitive)."""
    return df[df["nome_referencia"].str.contains("|".join(termos), case=False, na=False)]


def taxa_kinea(nome_contido: str) -> float:
    """Taxa de administração do fundo Kinea, vinda do pipeline principal (não do raw)."""
    linha = df_completo[df_completo["nome_padronizado"].str.contains(nome_contido, case=False, na=False)]
    return linha["taxa_administracao_pct"].iloc[0]


liquidez_custos = []

# --- Alpes Prev vs. 3 concorrentes diretos (SPX Seahawk, Ibiuna, Itaú Sinfonia) ---
conc_alpes = filtrar(df_conc_raw, "SPX Seahawk", "Ibiuna ST Prev", "Itau Sinfonia")
vals_alpes = _pct_to_float(conc_alpes["taxa_administracao"])
liquidez_custos.append({
    "fundo_kinea": "Kinea Alpes Prev XP Seg RF CP FICFI",
    "taxa_kinea_pct": taxa_kinea("Alpes Prev"),
    "mediana_concorrentes_pct": vals_alpes.median(),
    "n_concorrentes": vals_alpes.notna().sum(),
    "fonte": "scraper XP (concorrentes diretos identificados manualmente, ver config/concorrentes_aprofundamento.csv)",
    "observacao": "",
})

# --- KDIF11 vs. 3 concorrentes diretos (Sparta, Itaú Deb. Incentivadas, SulAmérica Infra) ---
conc_kdif = filtrar(df_conc_raw, "Sparta", "Itau Debentures", "SulAmerica")
vals_kdif = _pct_to_float(conc_kdif["taxa_administracao"])
liquidez_custos.append({
    "fundo_kinea": "Kinea Infra FII (KDIF11)",
    "taxa_kinea_pct": taxa_kinea("Infra"),
    "mediana_concorrentes_pct": vals_kdif.median(),
    "n_concorrentes": vals_kdif.notna().sum(),
    "fonte": "scraper XP (concorrentes diretos identificados manualmente, ver config/concorrentes_aprofundamento.csv)",
    "observacao": "",
})

# --- KNHY11 vs. universo amplo de concorrentes FII-Multicategoria (CVM) ---
LIMITE_MENSAL = 0.0025  # teto de sanidade: taxa de administração de FII raramente
                         # passa de ~0,25%/mês na prática de mercado; valores negativos
                         # ou acima disso são erro de preenchimento no informe mensal CVM

concorrentes_knhy11_amplo = df_concorrentes[
    (df_concorrentes["referencia_fundo_kinea"] == "Kinea High Yield CRI FII (KNHY11)")
    & (~df_concorrentes["eh_fundo_kinea"])
]
taxa_valida = concorrentes_knhy11_amplo["taxa_administracao_cvm_mensal_pct"].notna()
dentro_faixa = (
    (concorrentes_knhy11_amplo["taxa_administracao_cvm_mensal_pct"] >= 0)
    & (concorrentes_knhy11_amplo["taxa_administracao_cvm_mensal_pct"] <= LIMITE_MENSAL)
)
knhy_sane = concorrentes_knhy11_amplo[taxa_valida & dentro_faixa]
n_excluido = concorrentes_knhy11_amplo[taxa_valida].shape[0] - knhy_sane.shape[0]

mediana_mensal_knhy = knhy_sane["taxa_administracao_cvm_mensal_pct"].median()
mediana_anual_knhy = round(mediana_mensal_knhy * 1200, 2)

liquidez_custos.append({
    "fundo_kinea": "Kinea High Yield CRI FII (KNHY11)",
    "taxa_kinea_pct": taxa_kinea("High Yield"),
    "mediana_concorrentes_pct": mediana_anual_knhy,
    "n_concorrentes": len(knhy_sane),
    "fonte": "CVM inf_mensal_fii_complemento (Percentual_Despesas_Taxa_Administracao, "
             "despesa mensal reportada, anualizada x1200 como proxy - ver metodologia.md)",
    "observacao": f"Universo amplo (FII-Multicategoria): {len(concorrentes_knhy11_amplo)} concorrentes "
                  f"totais, {len(knhy_sane)} com taxa válida após filtro de sanidade "
                  f"(0-{LIMITE_MENSAL*100:.2f}%/mês; {n_excluido} excluídos por implausibilidade "
                  f"- negativo ou acima do teto, erro de preenchimento no informe mensal CVM).",
})

df_liquidez = pd.DataFrame(liquidez_custos)
df_liquidez["desvio_pct"] = (df_liquidez["taxa_kinea_pct"] - df_liquidez["mediana_concorrentes_pct"]).round(2)
df_liquidez["desvio_relativo_pct"] = ((df_liquidez["taxa_kinea_pct"] / df_liquidez["mediana_concorrentes_pct"] - 1) * 100).round(1)
df_liquidez["dimensao"] = "Liquidez e custos"

resultado_liquidez_view = df_liquidez[["fundo_kinea", "dimensao", "taxa_kinea_pct", "mediana_concorrentes_pct",
                                         "n_concorrentes", "desvio_relativo_pct", "fonte", "observacao"]]
print("=== Liquidez e custos ===")
display(resultado_liquidez_view)

resultado_liquidez_view.to_csv(PROJECT_ROOT / "data" / "processed" / "aprofundamento_liquidez_custos.csv", index=False)
print("\nSalvo em data/processed/aprofundamento_liquidez_custos.csv")

# %%
conteudo = [
    {"fundo_kinea": "Kinea Alpes Prev XP Seg RF CP FICFI",
     "achado": "Sem diferencial de conteúdo vs. concorrentes - template de Previdência da XP não expõe objetivo/benchmark para nenhum fundo (Kinea ou concorrente)",
     "evidencia": "Confirmado nas fichas de SPX Seahawk Icatu Prev e Ibiuna ST Prev Icatu, ambas sem seção Objetivo"},
    {"fundo_kinea": "Kinea Infra FII (KDIF11)",
     "achado": "Ficha 50% incompleta (falta quantidade_cotistas, valor_patrimonial) - concorrentes diretos EXPÕEM esses campos",
     "evidencia": "Devant (DEVA11): 84,1 mil cotistas, PL R$1,4bi | Valora (VGIP11): 84,3 mil cotistas, PL R$1bi - ambos visíveis na ficha pública"},
    {"fundo_kinea": "Kinea High Yield CRI FII (KNHY11)",
     "achado": "Sem achado - ficha 100% completa",
     "evidencia": "-"},
]
df_conteudo = pd.DataFrame(conteudo)
df_conteudo["dimensao"] = "Conteúdo e diferenciação"

resultado_conteudo_view = df_conteudo[["fundo_kinea", "dimensao", "achado", "evidencia"]]
print("=== Conteúdo e diferenciação ===")
display(resultado_conteudo_view)

resultado_conteudo_view.to_csv(PROJECT_ROOT / "data" / "processed" / "aprofundamento_conteudo.csv", index=False)
print("\nSalvo em data/processed/aprofundamento_conteudo.csv")

=== Liquidez e custos ===


,fundo_kinea,dimensao,taxa_kinea_pct,mediana_concorrentes_pct,n_concorrentes,desvio_relativo_pct,fonte,observacao
0,Kinea Alpes Prev XP Seg RF CP FICFI,Liquidez e custos,1.00,0.80,3,25.0,"scraper XP (concorrentes diretos identificados manualmente, ver config/concorrentes_aprofundamento.csv)",
1,Kinea Infra FII (KDIF11),Liquidez e custos,1.05,0.80,3,31.2,"scraper XP (concorrentes diretos identificados manualmente, ver config/concorrentes_aprofundamento.csv)",
2,Kinea High Yield CRI FII (KNHY11),Liquidez e custos,1.60,0.43,580,272.1,"CVM inf_mensal_fii_complemento (Percentual_Despesas_Taxa_Administracao, despesa mensal reportada, anualizada x1200 como proxy - ver metodologia.md)","Universo amplo (FII-Multicategoria): 647 concorrentes totais, 580 com taxa válida após filtro de sanidade (0-0.25%/mês; 67 excluídos por implausibilidade - negativo ou acima do teto, erro de preenchimento no informe mensal CVM)."



Salvo em data/processed/aprofundamento_liquidez_custos.csv
=== Conteúdo e diferenciação ===


,fundo_kinea,dimensao,achado,evidencia
0,Kinea Alpes Prev XP Seg RF CP FICFI,Conteúdo e diferenciação,Sem diferencial de conteúdo vs. concorrentes - template de Previdência da XP não expõe objetivo/benchmark para nenhum fundo (Kinea ou concorrente),"Confirmado nas fichas de SPX Seahawk Icatu Prev e Ibiuna ST Prev Icatu, ambas sem seção Objetivo"
1,Kinea Infra FII (KDIF11),Conteúdo e diferenciação,"Ficha 50% incompleta (falta quantidade_cotistas, valor_patrimonial) - concorrentes diretos EXPÕEM esses campos","Devant (DEVA11): 84,1 mil cotistas, PL R$1,4bi | Valora (VGIP11): 84,3 mil cotistas, PL R$1bi - ambos visíveis na ficha pública"
2,Kinea High Yield CRI FII (KNHY11),Conteúdo e diferenciação,Sem achado - ficha 100% completa,-



Salvo em data/processed/aprofundamento_conteudo.csv


# 6. Retorno e risco - consolidação

Janela de comparação: "No Ano", por ser a ÚNICA janela presente nas duas
estruturas de tabela da XP (`rr_*` de fundo_aberto/previdência tem
No Ano/12M/24M/36M/Desde o Início; `rent_*` de FII tem Dia/Semana/Mês/3M/6M/
No ano). Usar qualquer outra janela forçaria comparar fundo_aberto com FII
em bases inexistentes de um dos dois lados.

Exceção: KDIF11 - a ficha da XP para esse fundo está com a tabela de
Rentabilidade zerada (Fundo, IBOV e CDI todos "0%" nas 6 janelas -
confirmado no HTML bruto, não é bug do parser). A CVM também não expõe
rentabilidade de FII no Informe Mensal (só PL/cotistas/cadastro/balanço -
não é dado de preço de mercado). Usamos maisretorno.com como fonte
suplementar SÓ para Rentabilidade do KDIF11, na janela 12 Meses (única
disponível lá de forma limpa) - documentado explicitamente com URL e data
de acesso. Volatilidade e Sharpe do KDIF11 NÃO são comparados: a base de
cálculo (preço de bolsa, FII negociado) não é equivalente à dos
concorrentes diretos (valor de cota, fundos abertos não listados).

In [ ]:
def comparar_metricas(fundo_kinea: str, linha_kinea: pd.Series, df_concorrentes: pd.DataFrame,
                       metricas: list, fonte: str) -> list:
    """Monta as linhas de comparação Kinea vs. mediana dos concorrentes para
    uma lista de (rótulo_metrica, nome_da_coluna)."""
    linhas = []
    for rotulo, campo in metricas:
        valor_kinea = _pct_to_float(pd.Series([linha_kinea[campo]])).iloc[0]
        vals_conc = _pct_to_float(df_concorrentes[campo])
        linhas.append({
            "fundo_kinea": fundo_kinea,
            "dimensao": "Retorno e risco",
            "metrica": f"{rotulo} (No Ano)",
            "valor_kinea": valor_kinea,
            "mediana_concorrentes": vals_conc.median(),
            "n_concorrentes": vals_conc.notna().sum(),
            "fonte": fonte,
            "observacao": "",
        })
    return linhas


resultados = []

# --- Kinea Alpes Prev (previdencia) vs. 3 concorrentes diretos ---
kinea_alpes = filtrar(df_kinea_raw, "Alpes Prev").iloc[0]
conc_alpes_rr = filtrar(df_conc_raw, "SPX Seahawk", "Ibiuna ST Prev", "Itau Sinfonia")
resultados += comparar_metricas(
    "Kinea Alpes Prev XP Seg RF CP FICFI", kinea_alpes, conc_alpes_rr,
    metricas=[("Rentabilidade", "rr_rentabilidade_no_ano"),
              ("Volatilidade", "rr_volatilidade_no_ano"),
              ("Sharpe", "rr_sharpe_no_ano")],
    fonte="scraper XP - tabela 'Risco e Retorno'",
)

# --- Kinea High Yield CRI FII (KNHY11) vs. DEVA11 e VGIP11 ---
kinea_knhy = filtrar(df_kinea_raw, "High Yield").iloc[0]
conc_knhy_rr = filtrar(df_conc_raw, "Devant", "Valora")
resultados += comparar_metricas(
    "Kinea High Yield CRI FII (KNHY11)", kinea_knhy, conc_knhy_rr,
    metricas=[("Rentabilidade Fundo", "rent_fundo_no_ano"),
              ("Rentabilidade IBOV (benchmark)", "rent_ibov_no_ano"),
              ("Rentabilidade CDI (benchmark)", "rent_cdi_no_ano")],
    fonte="scraper XP - tabela 'Rentabilidade' (Fundo vs. IBOV vs. CDI)",
)

# --- Kinea Infra FII (KDIF11) - achado + Rentabilidade via maisretorno.com ---
conc_kdif_rr = filtrar(df_conc_raw, "Sparta", "Itau Debentures", "SulAmerica")
vals_rent_conc_kdif = _pct_to_float(conc_kdif_rr["rr_rentabilidade_12m"])

resultados.append({
    "fundo_kinea": "Kinea Infra FII (KDIF11)",
    "dimensao": "Retorno e risco",
    "metrica": "Rentabilidade (12 Meses)",
    "valor_kinea": 15.28,  # maisretorno.com, consultado 23/08/2026 - ver fontes.md
    "mediana_concorrentes": vals_rent_conc_kdif.median(),
    "n_concorrentes": vals_rent_conc_kdif.notna().sum(),
    "fonte": "maisretorno.com/fi-infra/kdif11 (fonte suplementar - ficha da XP para este "
             "fundo está com a tabela de Rentabilidade zerada/quebrada; ver observação)",
    "observacao": (
        "ACHADO: a ficha pública do KDIF11 na XP retorna Rentabilidade 0% em todas as "
        "6 janelas (Dia/Semana/Mês/3M/6M/No ano), inclusive nos benchmarks de mercado "
        "(IBOV, CDI) - confirmado no HTML bruto, não é bug de extração. A CVM não publica "
        "cotação/rentabilidade de FII no Informe Mensal (só PL, cotistas, cadastro e "
        "balanço). Usamos maisretorno.com como fonte suplementar só para Rentabilidade "
        "12M. Volatilidade e Sharpe NÃO comparados: KDIF11 é FII negociado em bolsa "
        "(volatilidade de preço de mercado); os concorrentes diretos são fundos abertos "
        "não listados (volatilidade de cota patrimonial) - bases de cálculo distintas, "
        "não comparáveis diretamente. Combinado com o achado já registrado em Conteúdo e "
        "diferenciação (ficha 50% incompleta), reforça um padrão de comunicação pública "
        "mais fraca para este fundo especificamente."
    ),
})

df_retorno_risco = pd.DataFrame(resultados)
df_retorno_risco["desvio_vs_mediana"] = (
    df_retorno_risco["valor_kinea"] - df_retorno_risco["mediana_concorrentes"]
).round(2)

pd.set_option("display.max_colwidth", None)
display(df_retorno_risco[["fundo_kinea", "metrica", "valor_kinea", "mediana_concorrentes",
                            "desvio_vs_mediana", "n_concorrentes"]])

df_retorno_risco.to_csv(PROJECT_ROOT / "data" / "processed" / "aprofundamento_retorno_risco.csv", index=False)
print("\nSalvo em data/processed/aprofundamento_retorno_risco.csv")

,fundo_kinea,metrica,valor_kinea,mediana_concorrentes,desvio_vs_mediana,n_concorrentes
0,Kinea Alpes Prev XP Seg RF CP FICFI,Rentabilidade (No Ano),8.35,8.74,-0.39,3
1,Kinea Alpes Prev XP Seg RF CP FICFI,Volatilidade (No Ano),0.23,1.78,-1.55,3
2,Kinea Alpes Prev XP Seg RF CP FICFI,Sharpe (No Ano),5.54,0.29,5.25,3
3,Kinea High Yield CRI FII (KNHY11),Rentabilidade Fundo (No Ano),-1.90,-7.38,5.48,2
4,Kinea High Yield CRI FII (KNHY11),Rentabilidade IBOV (benchmark) (No Ano),18.22,18.22,0.00,2
5,Kinea High Yield CRI FII (KNHY11),Rentabilidade CDI (benchmark) (No Ano),11.09,11.09,0.00,2
6,Kinea Infra FII (KDIF11),Rentabilidade (12 Meses),15.28,9.42,5.86,3



Salvo em data/processed/aprofundamento_retorno_risco.csv
